In [ ]:
pip install yfinance

In [1]:
import pandas as pd
import yfinance as yf
import time

def search_first_equity(company_name: str):
    """搜索公司名称，返回第一个 EQUITY 结果 股票代码 交易所 匹配名称"""
    if pd.isna(company_name) or company_name.strip() == "":
        return {"ticker": pd.NA, "exchange": pd.NA, "matched_name": pd.NA}

    try:
        search_results = yf.Search(query=company_name.strip(), max_results=10, news_count=0)
        quotes = getattr(search_results, "quotes", []) or []

        for q in quotes:
            qt = str(q.get("quoteType", "")).upper()
            td = str(q.get("typeDisp", "")).upper()
            if qt == "EQUITY" or td == "EQUITY":
                return {
                    "ticker": q.get("symbol", pd.NA),
                    "exchange": q.get("exchange") or q.get("exchDisp") or pd.NA,
                    "matched_name": q.get("longname") or q.get("shortname") or pd.NA
                }

        return {"ticker": pd.NA, "exchange": pd.NA, "matched_name": pd.NA}

    except Exception as e:
        print(f"搜索 {company_name} 出错: {e}")
        return {"ticker": pd.NA, "exchange": pd.NA, "matched_name": pd.NA}


def batch_match_tickers(df: pd.DataFrame, company_col="company", sleep_sec=0.2):
    """批量匹配 Ticker，返回增加列的 DataFrame"""
    tickers, exchanges, names = [], [], []

    for company in df[company_col]:
        result = search_first_equity(company)
        tickers.append(result["ticker"])
        exchanges.append(result["exchange"])
        names.append(result["matched_name"])
        time.sleep(sleep_sec)

    df = df.copy()
    df["ticker"] = tickers
    df["exchange"] = exchanges
    df["matched_name"] = names
    return df


In [3]:
if __name__ == "__main__":
    df = pd.read_csv("对原始合并数据进行180天去重后.csv")
    matched = batch_match_tickers(df)

In [4]:
print(matched)

       company                 location total_laid_off        date  \
0      Peloton            New York, USA        200-250  08/07/2025   
1     Siemens           Munich, Germany      2000-3000  03/18/2025   
2        Intel  Santa Clara, California         21,000  12/09/2025   
3       Amazon                  Seattle          14000  10/27/2025   
4        Nokia                    Espoo          14000  10/19/2023   
..         ...                      ...            ...         ...   
789      Zuora              SF Bay Area            NaN   12/6/2022   
790      Zuora              SF Bay Area            NaN   1/31/2024   
791       eBay       Tel Aviv, Non-U.S.            NaN   6/26/2024   
792  iSpecimen                   Boston            NaN    9/6/2023   
793      nCino               Wilmington            NaN   5/27/2025   

    announcement_date_verified  percentage_laid_off    industry  \
0                     2025/8/7                  NaN    Consumer   
1                    2025

In [6]:
matched.to_csv("C:\Users\31314\Desktop\layoffs_postipo_yahoo_matched.csv", index=False)

In [ ]:
import pandas as pd
import requests
import json
import time
import re
from datetime import datetime

def clean_company_name(name):
    """清理公司名称，移除常见后缀和特殊字符"""
    if pd.isna(name):
        return ""
    
    name = str(name).strip()
    
    # 移除常见后缀
    suffixes = [
        ' Inc.', ' Inc', ' Incorporated', ' Corp.', ' Corp', ' Corporation',
        ' Ltd.', ' Ltd', ' Limited', ' PLC', ' Plc', ' AG', ' SA', ' NV',
        ' Co.', ' Co', ' Company', ' Group', ' Holdings', ' Holding',
        ', Inc.', ', Inc', ', Corp.', ', Corp'
    ]
    
    for suffix in suffixes:
        if name.endswith(suffix):
            name = name[:-len(suffix)].strip()
    
    # 移除括号内容
    name = re.sub(r'\([^)]*\)', '', name)
    
    # 移除特殊字符但保留空格和基本标点
    name = re.sub(r'[^\w\s&.,-]', '', name)
    
    return name.strip()

def query_openfigi_enhanced(company_name, max_retries=2):
    """
    增强的OpenFIGI查询函数，使用多种策略
    """
    if not company_name or company_name == '':
        return None
    
    # 策略列表：尝试不同的查询方式
    strategies = [
        # 策略1: 使用清理后的完整公司名称
        {"idType": "NAME", "idValue": company_name},
        
        # 策略2: 使用原始名称（如果与清理后不同）
        # {"idType": "NAME", "idValue": original_name},
        
        # 策略3: 尝试使用可能的股票代码模式
        # 提取可能的首字母缩写
        {"idType": "TICKER", "idValue": ''.join([word[0] for word in company_name.split()[:3]])},
        
        # 策略4: 尝试只用第一个单词
        {"idType": "NAME", "idValue": company_name.split()[0] if company_name.split() else company_name},
    ]
    
    headers = {
        'Content-Type': 'application/json',
        'User-Agent': 'Python-OpenFIGI-Client/1.0'
    }
    
    for attempt in range(max_retries):
        for strategy in strategies:
            try:
                # 构建查询
                payload = [strategy]
                
                # 发送请求
                response = requests.post(
                    'https://api.openfigi.com/v3/mapping',
                    headers=headers,
                    data=json.dumps(payload),
                    timeout=10
                )
                
                if response.status_code == 200:
                    data = response.json()
                    
                    # 检查是否有有效结果
                    if data and len(data) > 0 and 'data' in data[0] and data[0]['data']:
                        # 过滤出股票类型的结果
                        equity_results = []
                        for item in data[0]['data']:
                            # 优先选择股票类型
                            if item.get('marketSector') == 'Equity':
                                equity_results.append(item)
                        
                        if equity_results:
                            # 取第一个股票结果
                            result = equity_results[0]
                            
                            # 获取交易所信息
                            exch_code = result.get('exchCode', '')
                            
                            # 交易所代码映射
                            exchange_map = {
                                'US': 'NYSE',
                                'NMS': 'NASDAQ',
                                'NYQ': 'NYSE',
                                'NGM': 'NASDAQ',
                                'ASE': 'NYSE American',
                                'PCX': 'NYSE Arca',
                                'BTS': 'BATS',
                                'LSE': 'London Stock Exchange',
                                'LON': 'London Stock Exchange',
                                'ASX': 'Australian Securities Exchange',
                                'TSE': 'Tokyo Stock Exchange',
                                'TYO': 'Tokyo Stock Exchange',
                                'HKG': 'Hong Kong Stock Exchange',
                                'SHA': 'Shanghai Stock Exchange',
                                'SHE': 'Shenzhen Stock Exchange',
                                'FRA': 'Frankfurt Stock Exchange',
                                'GER': 'Deutsche Börse',
                                'STO': 'Stockholm Stock Exchange',
                                'PAR': 'Euronext Paris',
                                'AMS': 'Euronext Amsterdam',
                                'BRU': 'Euronext Brussels',
                                'LIS': 'Euronext Lisbon',
                                'MCX': 'Moscow Exchange',
                                'TSX': 'Toronto Stock Exchange',
                                'TOR': 'Toronto Stock Exchange',
                                'VAN': 'TSX Venture Exchange',
                                'NSI': 'National Stock Exchange of India',
                                'BSE': 'Bombay Stock Exchange',
                                'NSE': 'National Stock Exchange of India',
                                'JSE': 'Johannesburg Stock Exchange',
                                'SAO': 'São Paulo Stock Exchange',
                                'SIN': 'Singapore Exchange',
                                'SGX': 'Singapore Exchange',
                                'KSC': 'Korea Exchange',
                                'TAI': 'Taiwan Stock Exchange',
                                'TWO': 'Taipei Exchange',
                            }
                            
                            exchange = exchange_map.get(exch_code, exch_code)
                            
                            return {
                                'ticker': result.get('ticker', ''),
                                'exchange': exchange,
                                'name': result.get('name', ''),
                                'compositeFIGI': result.get('compositeFIGI', ''),
                                'securityType': result.get('securityType', ''),
                                'marketSector': result.get('marketSector', ''),
                                'exchCode': exch_code
                            }
                
                time.sleep(0.5)  # 请求间隔
                
            except requests.exceptions.RequestException as e:
                print(f"请求失败: {e}")
                continue
            except json.JSONDecodeError as e:
                print(f"JSON解析失败: {e}")
                continue
        
        # 如果所有策略都失败，等待后重试
        if attempt < max_retries - 1:
            wait_time = 2 ** attempt
            print(f"等待 {wait_time} 秒后重试...")
            time.sleep(wait_time)
    
    return None

def main():
    # 1. 读取数据
    print("读取数据文件...")
    try:
        # 尝试不同的编码
        df = pd.read_csv('C:/Users/31314/Desktop/layoffs_postipo_yahoo_matched.csv', encoding='utf-8')
    except:
        try:
            df = pd.read_csv('C:/Users/31314/Desktop/layoffs_postipo_yahoo_matched.csv', encoding='gbk')
        except:
            df = pd.read_csv('C:/Users/31314/Desktop/layoffs_postipo_yahoo_matched.csv', encoding='latin1')
    
    print(f"读取成功，共 {len(df)} 条记录")
    
    # 2. 识别需要查询的公司
    # 条件：ticker为空、NaN，或者包含"XYZ"
    def needs_openfigi(row):
        ticker = str(row['ticker']).strip() if 'ticker' in row and pd.notna(row['ticker']) else ''
        return (ticker == '' or ticker.lower() == 'nan' or 'xyz' in ticker.lower())
    
    df['needs_openfigi'] = df.apply(needs_openfigi, axis=1)
    companies_to_query = df[df['needs_openfigi']]
    
    print(f"需要查询OpenFIGI的公司数量: {len(companies_to_query)}")
    print(f"占比: {len(companies_to_query)/len(df)*100:.1f}%")
    
    if len(companies_to_query) == 0:
        print("没有需要查询的公司")
        return
    
    # 3. 初始化新列（如果不存在）
    if 'openfigi_ticker' not in df.columns:
        df['openfigi_ticker'] = ''
    if 'openfigi_exchange' not in df.columns:
        df['openfigi_exchange'] = ''
    if 'openfigi_matched_name' not in df.columns:
        df['openfigi_matched_name'] = ''
    if 'openfigi_composite_figi' not in df.columns:
        df['openfigi_composite_figi'] = ''
    if 'data_source' not in df.columns:
        df['data_source'] = 'yahoo'  # 默认为yahoo
    
    # 4. 批量查询OpenFIGI
    print("\n开始查询OpenFIGI API...")
    print("="*50)
    
    success_count = 0
    fail_count = 0
    total = len(companies_to_query)
    
    for idx, row in companies_to_query.iterrows():
        company = row['company']
        original_ticker = row['ticker'] if 'ticker' in row else ''
        
        if pd.isna(company):
            print(f"[{success_count+fail_count+1}/{total}] 跳过: 公司名称为空")
            fail_count += 1
            continue
        
        # 清理公司名称
        cleaned_name = clean_company_name(company)
        
        print(f"[{success_count+fail_count+1}/{total}] 查询: {company}")
        if cleaned_name != company:
            print(f"      清理后: {cleaned_name}")
        
        # 查询OpenFIGI
        result = query_openfigi_enhanced(cleaned_name)
        
        if result:
            # 更新数据
            df.at[idx, 'openfigi_ticker'] = result['ticker']
            df.at[idx, 'openfigi_exchange'] = result['exchange']
            df.at[idx, 'openfigi_matched_name'] = result['name']
            df.at[idx, 'openfigi_composite_figi'] = result['compositeFIGI']
            df.at[idx, 'data_source'] = 'openfigi'
            
            # 如果原始ticker是XYZ或空，用OpenFIGI的结果更新主ticker列
            if needs_openfigi(row):
                df.at[idx, 'ticker'] = result['ticker']
                df.at[idx, 'exchange'] = result['exchange']
                df.at[idx, 'matched_name'] = result['name']
            
            success_count += 1
            print(f"  ✓ 匹配成功: {result['ticker']} ({result['exchange']})")
            print(f"     名称: {result['name']}")
            print(f"     类型: {result.get('securityType', 'N/A')}")
        else:
            fail_count += 1
            print(f"  ✗ 未找到匹配")
        
        # 显示进度
        if (success_count + fail_count) % 10 == 0:
            print(f"进度: {success_count+fail_count}/{total} (成功: {success_count}, 失败: {fail_count})")
        
        # 避免请求过快
        time.sleep(1)
    
    # 5. 保存结果
    print("\n" + "="*50)
    print("保存结果...")
    
    # 保存完整结果
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = f'C:/Users/31314/Desktop/layoffs_openfigi_supplemented_{timestamp}.csv'
    df.to_csv(output_file, index=False, encoding='utf-8')
    
    # 6. 生成报告
    print("\n" + "="*50)
    print("处理完成！统计报告")
    print("="*50)
    
    # 统计最终匹配情况
    yahoo_matched = df[df['data_source'] == 'yahoo']
    openfigi_matched = df[df['data_source'] == 'openfigi']
    still_unmatched = df[df['needs_openfigi'] & (df['data_source'] != 'openfigi')]
    
    print(f"原始数据总数: {len(df)}")
    print(f"Yahoo Finance匹配: {len(yahoo_matched)}")
    print(f"OpenFIGI补充匹配: {len(openfigi_matched)}")
    print(f"仍未匹配: {len(still_unmatched)}")
    print(f"\nOpenFIGI查询成功率: {success_count}/{total} ({success_count/total*100:.1f}%)")
    
    # 显示OpenFIGI匹配的示例
    if len(openfigi_matched) > 0:
        print(f"\nOpenFIGI匹配示例 (前10个):")
        samples = openfigi_matched.head(10)
        for _, row in samples.iterrows():
            print(f"  {row['company']} -> {row['ticker']} ({row['exchange']})")
    
    # 显示仍未匹配的示例
    if len(still_unmatched) > 0:
        print(f"\n仍未匹配的公司示例 (前10个):")
        samples = still_unmatched.head(10)
        for _, row in samples.iterrows():
            print(f"  {row['company']}")
    
    print(f"\n结果已保存到: {output_file}")
    
    # 7. 保存仍未匹配的公司列表（用于手动处理）
    if len(still_unmatched) > 0:
        unmatched_file = f'still_unmatched_{timestamp}.csv'
        still_unmatched[['company', 'location', 'industry']].to_csv(unmatched_file, index=False, encoding='utf-8')
        print(f"仍未匹配的公司列表已保存到: {unmatched_file}")
        
        print("\n对于仍未匹配的公司，建议:")
        print("1. 检查公司名称的准确性")
        print("2. 尝试使用公司的官方全称")
        print("3. 手动搜索确认公司是否已上市")
        print("4. 考虑使用其他数据源（如彭博终端、路透社等）")

if __name__ == "__main__":
    # 安装依赖（如果尚未安装）
    try:
        import pandas as pd
        import requests
    except ImportError:
        print("请先安装所需依赖:")
        print("pip install pandas requests")
    
    # 运行主程序
    main()